# 03 — Propensity Score Matching

Goal: Estimate ATT via nearest-neighbor matching on propensity scores.

Steps:
1. Fit propensity model (P(treatment | confounders))
2. Check common support (overlap region)
3. Match treated to control on propensity score
4. Check covariate balance before/after (SMD plot)
5. Estimate ATT with bootstrap CI
6. Sensitivity analysis: Rosenbaum bounds (how much unobserved confounding would overturn result?)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import sys; sys.path.insert(0, '..')
from src.estimators import propensity_score_matching
from src.diagnostics import plot_propensity_balance

df = pd.read_csv('../data/field_panel.csv', parse_dates=['install_date'])
df['install_month_num'] = df.install_date.dt.month + (df.install_date.dt.year - 2022) * 12
CONFOUNDERS = ['install_month_num', 'operating_hours_month']

%matplotlib inline

In [ ]:
# Fit propensity model
df_enc = pd.get_dummies(df[CONFOUNDERS + ['region', 'install_crew']], drop_first=True)
X = df_enc.values
y = (df.design_variant == 'B').astype(int).values

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X, y)
df['propensity_score'] = lr.predict_proba(X)[:, 1]

print(f'Propensity score range: {df.propensity_score.min():.3f} – {df.propensity_score.max():.3f}')

In [ ]:
# Common support check — histogram of propensity scores by treatment
fig, ax = plt.subplots(figsize=(8, 4))
for grp, color, label in [('A', 'tomato', 'Control (A)'), ('B', 'steelblue', 'Treated (B)')]:
    sub = df[df.design_variant == grp].propensity_score
    ax.hist(sub, bins=40, alpha=0.55, color=color, label=label, density=True)
ax.set_title('Propensity Score Distribution — Check for Common Support')
ax.set_xlabel('Propensity Score P(Variant B)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/propensity_overlap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Common support: overlap region between two histograms')

In [ ]:
# Run propensity score matching (from src/estimators.py)
result = propensity_score_matching(df, CONFOUNDERS + ['region', 'install_crew'])
print(result)

In [ ]:
# Bootstrap CI for ATT
np.random.seed(42)
boot_effects = []
for _ in range(500):
    sample = df.sample(frac=1, replace=True)
    r = propensity_score_matching(sample, CONFOUNDERS + ['region', 'install_crew'])
    boot_effects.append(r['effect_estimate_pct'])

ci_lower, ci_upper = np.percentile(boot_effects, [2.5, 97.5])
print(f'ATT: {result["effect_estimate_pct"]:+.1f}% (95% CI: [{ci_lower:.1f}%, {ci_upper:.1f}%])')
print(f'True effect: -15.0%')
print(f'Recovered? {ci_lower < -15.0 < ci_upper}')

In [ ]:
# Sensitivity analysis note
print('Rosenbaum bounds: How strong must unobserved confounding be to explain away the result?')
print('Compute manually or use the R rbounds package / Python approximation.')
print('Document the E-value: minimum confounder effect size that could nullify the result.')
print('Even if not computed, *mentioning* this in the report shows causal-inference maturity.')